# MEL — Smoke test QLoRA 4-bit
Ce notebook exécute un vrai smoke test GPU de 500 conversations. Il échoue immédiatement si aucun GPU CUDA n'est disponible. Le résultat attendu est `TRAINED_UNBENCHMARKED`, jamais une promotion automatique.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU CUDA requis : activez un runtime GPU (T4/L4/A100) puis relancez Run all.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!rm -rf /content/meliturgos-cloudflare
!git clone --branch candidate/mel-clean-autonomy --single-branch https://github.com/adrienlopezcarreras-pixel/meliturgos-cloudflare.git /content/meliturgos-cloudflare
%cd /content/meliturgos-cloudflare
!python -m pip install -U pip
!python -m pip install -r requirements-lora.txt ijson

In [ ]:
!mkdir -p artifacts/lora-data artifacts/lora-train
!python scripts/prepare-sharegpt-lora.py --variant both --max-conversations 500 --seed 42 --output artifacts/lora-data/sharegpt-mel-smoke-500.jsonl

In [ ]:
!python scripts/train-mel-lora.py --dataset artifacts/lora-data/sharegpt-mel-smoke-500.jsonl --output artifacts/lora-train/smoke-500 --epochs 1 --save-steps 50 --seed 42

In [ ]:
from pathlib import Path
import json
root = Path('artifacts/lora-train/smoke-500')
required = [root/'adapter_model.safetensors', root/'adapter_config.json', root/'training-evidence.json']
for p in required:
    assert p.is_file() and p.stat().st_size > 0, f'MISSING: {p}'
e = json.loads((root/'training-evidence.json').read_text())
assert e['status'] == 'TRAINED_UNBENCHMARKED'
assert e['training_mode'] == 'qlora-4bit-nf4'
assert e['quantization']['bits'] == 4
assert e['environment']['cuda_available'] is True
print(json.dumps({'status':'PASS','gpu':e['environment']['gpu'],'examples':e['dataset']['examples'],'train_loss':e['training_metrics']['train_loss'],'global_step':e['training_metrics']['global_step']}, indent=2))

In [ ]:
!cd artifacts/lora-train && zip -r /content/MEL-QLORA-smoke-500.zip smoke-500 >/dev/null
print('Artefact prêt : /content/MEL-QLORA-smoke-500.zip')